# Module 7: Deploy to AgentCore Runtime (15 min)

Deploy the customer service agent to **Bedrock AgentCore Runtime**. Package it as a service and invoke it from the CLI.

**Prerequisites:** Modules 1-6 completed, AWS credentials with AgentCore access

---

## Step 1: Install the AgentCore CLI

The `bedrock-agentcore-starter-toolkit` package provides the `agentcore` CLI for packaging and deploying agents. The `bedrock-agentcore` package is the runtime SDK your agent imports (`BedrockAgentCoreApp`).

In [ ]:
!pip install -q bedrock-agentcore-starter-toolkit bedrock-agentcore strands-agents

---

## Step 2: Configure the Agent

Your agent already lives in `main.py` (it imports the tools and steering handlers from this folder). Run `agentcore configure` to prepare it for deployment — this points the CLI at your entrypoint and generates the deployment files.

In [ ]:
# Configure the agent for deployment (run in your terminal from this folder)
# !agentcore configure -e main.py

print("""
Run from this folder:

  agentcore configure -e main.py

The -e/--entrypoint flag points the CLI at main.py. This records your
deployment settings in .bedrock_agentcore.yaml. The default build type is
direct_code_deploy: your code is zipped and uploaded to S3 (no container).
""")

---

## Step 3: The Deployment Code

Replace the generated `main.py` with our customer service agent. The pattern is:
1. Create a `BedrockAgentCoreApp`
2. Define an `@app.entrypoint` that receives `payload` and `context`
3. Call `app.run()` at the bottom

In [ ]:
# Let's look at our main.py
print(open("main.py").read())

`configure` packages this folder, so `main.py`, `customer_service_tools.py`, and `steering_handlers.py` are included automatically.

Make sure `requirements.txt` lists the runtime dependencies:
```
strands-agents>=1.42.0
bedrock-agentcore>=1.13.0
```

---

## Step 3: Deploy to AgentCore Runtime

`agentcore deploy` packages your code, uploads it to an Amazon S3 staging bucket, and provisions the AgentCore Runtime via AWS CloudFormation. Run it from the same folder, after `configure`.

In [ ]:
# Deploy the agent (run in your terminal after configure)
# !agentcore deploy

print("""
Run from this folder:

  agentcore deploy

This packages your code, uploads it to an S3 staging bucket, and provisions
the AgentCore Runtime via AWS CloudFormation.

Prerequisites:
  1. AWS credentials configured
  2. agentcore configure already run (.bedrock_agentcore.yaml present)

The default direct_code_deploy build runs in the cloud — no Docker needed.
""")

---

## Step 4: Invoke the Deployed Agent

`agentcore invoke` sends a request to your deployed agent. The payload is a JSON string whose `prompt` field reaches your `@app.entrypoint`.

In [ ]:
# Invoke the deployed agent (run in your terminal)
# !agentcore invoke '{"prompt": "Hi, I am customer C-1001. What are my recent orders?"}'

# Target a session for multi-turn conversations.
# NOTE: --session-id must be 33-256 characters, so pad short IDs:
# !agentcore invoke '{"prompt": "I need a refund for order ORD-5521"}' --session-id customer-1001-session-abcdefghijklmnop

print("""
Run from this folder:

  agentcore invoke '{"prompt": "Hi, I am customer C-1001. What are my recent orders?"}'

The payload is a positional JSON string. Use --session-id to keep context
across calls - it must be 33-256 characters (a short value like 'customer-1001'
will fail), e.g. 'customer-1001-session-abcdefghijklmnop'.
""")

---

## Local Testing

You can test the agent locally before deploying:

In [ ]:
# For local testing without AgentCore, run the agent directly:
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager
from customer_service_tools import lookup_customer, get_order_history, process_refund
from steering_handlers import RefundWorkflowHandler, tone_handler

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[RefundWorkflowHandler(), tone_handler],
    system_prompt="You are a customer service agent. Be helpful and concise.",
    conversation_manager=SlidingWindowConversationManager(window_size=20),
)

result = agent("Hi, I'm customer C-1001. What are my recent orders?")
print(f"\n✅ Agent responded. Tokens used: {result.metrics.accumulated_usage['totalTokens']}")

---

## 🎓 Workshop Complete!

You've built a production-ready customer service agent from scratch:

1. ✅ **Agent Loop + Tools** — Core agent with customer service capabilities
2. ✅ **Hooks** — Rate limiting via deterministic code in the loop
3. ✅ **Skills + Steering** — Workflow knowledge and business rule enforcement
4. ✅ **Session Managers** — Persistent memory across restarts
5. ✅ **Multi-Agent** — Delegation to specialist agents
6. ✅ **Evals** — Automated testing of output quality and tool trajectories
7. ✅ **Deploy** — Production deployment with AgentCore Runtime

## 📚 Resources

- [Strands Agents Documentation](https://strandsagents.com)
- [Strands GitHub](https://github.com/strands-agents/sdk-python)
- [AgentCore Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/)
- [AgentCore Starter Toolkit](https://aws.github.io/bedrock-agentcore-starter-toolkit/)